# 93 Veeva Link review pack (Phase 3: B3)

Review pack for Field Analytics and Medical Affairs (`90_` task B3), before any rep-facing use (P3-D1). Per list / hint: 15 sampled HCPs (fixed seed) with their Veeva evidence and empty feedback columns (`REVIEW_AGREE` Y/N, `REVIEW_REASON`, `REVIEWER`). Payment columns are never included (P3-D6).

Inputs: `91_` run `20261123T103200` (B0, B1, B4), `92_` run `20261123T103700` (B2). Outputs: `OUT/business/review_pack/run=<ts>/{pack, pack_csv, cover}`.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np, datetime as dt

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
PB = OUT + "business/"
R91, R92 = "20261123T103200", "20261123T103700"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
N = 15

rd = lambda p: spark.read.parquet(p).toPandas()
b1 = rd(f"{PB}b1_untargeted_authority/run={R91}/list")
b4 = rd(f"{PB}b4_low_adoption_idn_experts/run={R91}/list")
b2 = rd(f"{PB}b2_action_hints/run={R92}/list")
EVID = ["PRIMARY_HINT", "SECONDARY_HINTS", "CLINICAL_EVIDENCE_PI", "veeva_event_podium_flag_3y", "veeva_event_n_inscope_3y", "veeva_trial_n_inscope_5y",
        "veeva_trial_pi_flag_5y", "veeva_trial_finerenone_flag_5y", "veeva_trial_mra_flag_5y", "veeva_pub_n_inscope_5y", "veeva_dr_any_inscope"]
ev = b2[["NPI_NUMBER"] + EVID]
KEEP = ["REVIEW_GROUP", "NPI_NUMBER", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE", "SEGMENT", "veeva_best_decile", "TIER",
        "INDICATIONS", "KER10_WRITER_12M", "IDN_NAME", "IDN_PEER_WRITER_SHARE", *EVID, "veeva_score_from_stale_load"]

groups = []
for tier in ("Top (9-10)", "High (8)"):
    groups.append(b1[b1.TIER == tier].sample(N, random_state=7).assign(REVIEW_GROUP=f"B1 untargeted authority, {tier}"))
groups.append(b4.sample(N, random_state=7).assign(REVIEW_GROUP="B4 low-adoption IDN expert"))
for h in ["P2P_SPEAKER", "CLINICAL_EVIDENCE", "CARDIORENAL_CLASS", "DISEASE_CKD", "DISEASE_HF", "DISEASE_T2D", "SCIENTIFIC_CONTENT"]:
    groups.append(b2[b2.PRIMARY_HINT == h].sample(N, random_state=7).drop(columns=EVID[1:]).assign(REVIEW_GROUP=f"B2 hint {h}"))
pack = pd.concat(groups, ignore_index=True)
pack = pack.drop(columns=[c for c in EVID if c in pack.columns]).merge(ev, on="NPI_NUMBER", how="left")
for c in KEEP:
    if c not in pack.columns:
        pack[c] = None
pack = pack[KEEP]
pack["REVIEW_AGREE"], pack["REVIEW_REASON"], pack["REVIEWER"] = "", "", ""

cover = pd.DataFrame([
    {"item": "Status", "text": "Offline review only. Not for rep use until licence and compliance answers (Q1, Q2) are in."},
    {"item": "Data age", "text": "Veeva Link scores and activity end 2025-03-06 (about 19 months before the list date)."},
    {"item": "B1", "text": f"{len(b1):,} Non-target HCPs at Veeva best decile 8-10. Writer rate for Non-target at decile 8-10 was 0.52% (C2, R4): a list for target review, not a list of likely writers."},
    {"item": "B4", "text": f"{len(b4):,} non-writing experts in {b4.L4_ID.nunique():,} IDNs where 5% or less of colleagues wrote Kerendia in the last 13 weeks."},
    {"item": "B2", "text": f"{len(b2):,} HCPs with an action hint; one primary hint each by fixed priority. Hints are rules on Veeva activity and rankings, not model scores."},
    {"item": "Please check", "text": "Is the HCP correctly described (specialty, activity)? Is the hint sensible? Any conflict with Bayer speaker or MSL knowledge?"}])

P = f"{PB}review_pack/run={RUN_TS}/"
x = pack.astype(str).replace({"nan": None, "None": None})
spark.createDataFrame(x).write.mode("overwrite").parquet(P + "pack")
spark.createDataFrame(x).coalesce(1).write.mode("overwrite").option("header", True).csv(P + "pack_csv")
spark.createDataFrame(cover).write.mode("overwrite").parquet(P + "cover")

chk = pd.DataFrame([
    {"check": "P1", "name": "rows per group = 15", "value": pack.REVIEW_GROUP.value_counts().to_dict(), "pass": bool((pack.REVIEW_GROUP.value_counts() == N).all())},
    {"check": "M6", "name": "no payment column in pack", "value": [c for c in pack.columns if "pay" in c.lower()], "pass": not any("pay" in c.lower() for c in pack.columns)},
    {"check": "P2", "name": "every pack NPI found in its source list", "value": int((~pack.NPI_NUMBER.isin(set(b1.NPI_NUMBER) | set(b4.NPI_NUMBER.dropna()) | set(b2.NPI_NUMBER))).sum()),
     "pass": bool(pack.NPI_NUMBER.isin(set(b1.NPI_NUMBER) | set(b4.NPI_NUMBER.dropna()) | set(b2.NPI_NUMBER)).all())}])
chk["value"] = chk.value.astype(str); chk["run_ts"] = RUN_TS
spark.createDataFrame(chk.astype(str)).write.mode("overwrite").parquet(f"{PB}checks/run={RUN_TS}/b3")
print(f"pack rows {len(pack)} in {pack.REVIEW_GROUP.nunique()} groups | written: {P}")
display(chk)
display(pack.head(5))